# 06. Neo4j 인용/참조 필드 분석

**목표**: 11개 카테고리에서 그래프 DB에 적합한 인용/참조/관계 데이터를 식별

**분석 내용**:
1. 인용/참조 필드 인벤토리 (구조화 vs 비구조화)
2. 구조화된 인용 필드 분석 (참조조문, 참조판례, 관련법령)
3. 인용 포맷 패턴 분석 (정규식 추출)
4. 비구조화 텍스트 내 법령 참조 탐색
5. 교차 참조 가능성 + Neo4j 확장 우선순위

**의존**: `eda_output/phase1_inventory.json`, `eda_output/phase2_schema.json`

**산출물**: `eda_output/phase6_citation_analysis.json`

In [1]:
# ── Cell 1: 환경 설정 ──────────────────────────────────────
import sys
from pathlib import Path

BACKEND_DIR = Path.cwd().parent.parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from scripts.eda.common import (
    DATA_DIR, load_all, head_sample,
    load_result, save_result,
    extract_citations, extract_law_names,
)
from scripts.eda.data_registry import CATEGORIES

pio.templates.default = "plotly_white"

# ── 데이터 모드 토글 ──────────────────────────────────────
# True  = 전체 데이터 (모든 레코드, 시간 + 메모리 소요)
# False = 샘플 데이터 (head_sample 1000건, 빠른 미리보기)
USE_FULL_DATA = True
SAMPLE_N = 1000  # 샘플 모드일 때 건수

# 이전 결과 로드
phase1 = load_result("phase1_inventory")
phase2 = load_result("phase2_schema")

# 카테고리별 레코드 수
cat_records = {}
for f in phase1:
    cat = f["category"]
    cat_records[cat] = cat_records.get(cat, 0) + f["record_count"]

mode = "전체 데이터 (모든 레코드)" if USE_FULL_DATA else f"샘플 데이터 (head_sample {SAMPLE_N}건)"
print(f"데이터 모드: {mode}")
print(f"카테고리 수: {len(CATEGORIES)}")

데이터 모드: 전체 데이터 (모든 레코드)
카테고리 수: 11


In [2]:
# ── Cell 2: 인용/참조 필드 인벤토리 ──────────────────────
# 인용/참조 관련 필드 키워드
CITATION_KEYWORDS = ["참조", "인용", "관련법령", "심판대상조문", "법령"]

inventory_data = []

for cat_key, cat_info in CATEGORIES.items():
    cat_label = cat_info["label"]
    schema = phase2.get(cat_key, {})
    fields = schema.get("fields", {})
    
    citation_fields = []
    for field_name, field_info in fields.items():
        # 인용/참조 관련 필드인지 확인
        is_citation = any(kw in field_name for kw in CITATION_KEYWORDS)
        if not is_citation:
            continue
        
        empty_rate = field_info.get("empty_rate", 1.0)
        presence = field_info.get("presence_rate", 0)
        non_empty_rate = round(presence * (1 - empty_rate) * 100, 1)
        
        citation_fields.append({
            "field": field_name,
            "non_empty_rate": non_empty_rate,
            "empty_rate": round(empty_rate * 100, 1),
        })
    
    # 인벤토리에 추가
    if citation_fields:
        for cf in citation_fields:
            inventory_data.append({
                "카테고리": cat_label,
                "category": cat_key,
                "인용 필드": cf["field"],
                "유효 데이터율 (%)": cf["non_empty_rate"],
                "비어있는 비율 (%)": cf["empty_rate"],
                "레코드 수": cat_records.get(cat_key, 0),
                "유형": "구조화",
            })
    else:
        inventory_data.append({
            "카테고리": cat_label,
            "category": cat_key,
            "인용 필드": "(없음 - 텍스트 파싱 필요)",
            "유효 데이터율 (%)": 0.0,
            "비어있는 비율 (%)": 100.0,
            "레코드 수": cat_records.get(cat_key, 0),
            "유형": "비구조화",
        })

df_inv = pd.DataFrame(inventory_data)
display(df_inv.style.format({
    "유효 데이터율 (%)": "{:.1f}",
    "비어있는 비율 (%)": "{:.1f}",
    "레코드 수": "{:,}",
}))

# 히트맵: 카테고리별 인용 필드 존재 여부
structured = df_inv[df_inv["유형"] == "구조화"]
if len(structured) > 0:
    pivot = structured.pivot_table(
        index="카테고리", columns="인용 필드", values="유효 데이터율 (%)", fill_value=0
    )
    fig = px.imshow(
        pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        title="구조화된 인용 필드 매트릭스 (유효 데이터율 %)",
        color_continuous_scale="YlOrRd",
        text_auto=".1f",
        aspect="auto",
    )
    fig.update_layout(height=400)
    fig.show()

,카테고리,category,인용 필드,유효 데이터율 (%),비어있는 비율 (%),레코드 수,유형
0,판례,precedent,참조조문,81.3,18.7,"92,055",구조화
1,판례,precedent,참조판례,44.0,56.0,"92,055",구조화
2,법령,law,법령ID,100.0,0.0,"5,548",구조화
3,법령,law,법령명_한글,100.0,0.0,"5,548",구조화
4,법령,law,법령 요약,100.0,0.0,"5,548",구조화
5,헌재결정례,constitutional,심판대상조문,10.6,89.4,"31,718",구조화
6,헌재결정례,constitutional,참조판례,13.1,86.9,"31,718",구조화
7,헌재결정례,constitutional,참조조문,14.2,85.8,"31,718",구조화
8,행정심판례,administration,(없음 - 텍스트 파싱 필요),0.0,100.0,"34,254",비구조화
9,특별행정심판,special_tribunal,(없음 - 텍스트 파싱 필요),0.0,100.0,"148,778",비구조화


In [3]:
# ── Cell 3: 구조화된 인용 필드 분석 (High Priority) ───────
# 구조화된 인용 필드가 있는 카테고리만 상세 분석

STRUCTURED_TARGETS = {
    "precedent": ["참조조문", "참조판례"],
    "constitutional": ["참조조문", "참조판례", "심판대상조문"],
    "cgm_expc": ["관련법령"],
}

structured_analysis = []

for cat_key, fields in STRUCTURED_TARGETS.items():
    cat_info = CATEGORIES[cat_key]
    cat_label = cat_info["label"]
    first_file = cat_info["files"][0]
    file_path = DATA_DIR / first_file
    
    if not file_path.exists():
        print(f"  ⏭️ {cat_label}: 파일 없음")
        continue
    
    if USE_FULL_DATA:
        sample = load_all(file_path)
    else:
        sample = head_sample(file_path, n=SAMPLE_N)
    
    print(f"\n{'='*50}")
    print(f"{cat_label} ({cat_key}) - {len(sample)}건 {'전체' if USE_FULL_DATA else '샘플'}")
    print(f"{'='*50}")
    
    for field_name in fields:
        non_empty = []
        for rec in sample:
            val = rec.get(field_name, "")
            if isinstance(val, str) and val.strip():
                non_empty.append(val)
        
        rate = len(non_empty) / len(sample) * 100 if sample else 0
        
        # 실제 값 샘플 (최대 3개)
        samples_text = non_empty[:3]
        
        # 인용 추출 시도
        citation_counts = []
        for text in non_empty:
            citations = extract_citations(text)
            law_names = extract_law_names(text)
            # 관련법령 필드는 전체가 법령명인 경우가 많음
            total_refs = len(citations) + len(law_names)
            citation_counts.append(max(total_refs, 1) if text.strip() else 0)
        
        avg_citations = np.mean(citation_counts) if citation_counts else 0
        
        result_entry = {
            "카테고리": cat_label,
            "category": cat_key,
            "필드": field_name,
            "유효 건수": len(non_empty),
            "유효율 (%)": round(rate, 1),
            "평균 인용 수": round(avg_citations, 1),
        }
        structured_analysis.append(result_entry)
        
        print(f"\n  [{field_name}] 유효: {len(non_empty)}/{len(sample)} ({rate:.1f}%), 평균 인용: {avg_citations:.1f}")
        for i, s in enumerate(samples_text[:2]):
            preview = s[:120].replace("\n", " ")
            print(f"    샘플 {i+1}: {preview}...")

df_structured = pd.DataFrame(structured_analysis)
display(df_structured.style.format({
    "유효 건수": "{:,}",
    "유효율 (%)": "{:.1f}",
    "평균 인용 수": "{:.1f}",
}))


판례 (precedent) - 92055건 전체

  [참조조문] 유효: 73971/92055 (80.4%), 평균 인용: 1.7
    샘플 1: 민법 제750조, 제756조...
    샘플 2: 민법 제536조, 제568조...

  [참조판례] 유효: 41952/92055 (45.6%), 평균 인용: 1.0
    샘플 1: 1980. 4. 22. 선고 80다268 판결(요추 II 민법 제536조(1)41면 집28① 민 247 공634호 12808카12379)...
    샘플 2: 1967.12.19. 선고 67다1694 판결(판례카아드 2180호, 대법원판결집 15③민395, 판결요지집 산림법 제34조(1) 1724면)...

헌재결정례 (constitutional) - 31718건 전체

  [참조조문] 유효: 4567/31718 (14.4%), 평균 인용: 1.6
    샘플 1: 헌법 제11조 제1항, 제39조 제2항헌법재판소법 제68조 제1항헌법재판소법 제40조(준용규정) ① 헌법재판소의 심판절차에 관하여는 이 법에 특별한 규정이 있는 경우를 제외하고는 헌법재판의 성질에 반하지 아니하는 한...
    샘플 2: 가. 헌법재판소법(憲法裁判所法) 제68조 제1항, 제71조 제1항나. 헌법재판소법(憲法裁判所法) 제68조 제1항다. 헌법재판소법(憲法裁判所法) 제69조 제1항민사소송법(民事訴訟法) 제160조행정심판법(行政審判法) 제...

  [참조판례] 유효: 4312/31718 (13.6%), 평균 인용: 1.0
    샘플 1: 2. 헌재 1998. 9. 30. 96헌바88, 판례집 10-2, 517, 529헌재 2002. 12. 18. 2001헌마111, 판례집 14-2, 872, 8793. 헌재 1993. 9. 27. 89헌마248, 판...
    샘플 2: 91헌마19089헌마3188헌마2291헌마19089헌마220...

  [심판대상조문] 유효: 3650/31718 (11.5%), 평균 인용: 1.1
    샘플 1: 1980년해직

,카테고리,category,필드,유효 건수,유효율 (%),평균 인용 수
0,판례,precedent,참조조문,"73,971",80.4,1.7
1,판례,precedent,참조판례,"41,952",45.6,1.0
2,헌재결정례,constitutional,참조조문,"4,567",14.4,1.6
3,헌재결정례,constitutional,참조판례,"4,312",13.6,1.0
4,헌재결정례,constitutional,심판대상조문,"3,650",11.5,1.1
5,부처 해석례,cgm_expc,관련법령,528,100.0,1.5


In [4]:
# ── Cell 4: 인용 포맷 패턴 분석 ──────────────────────────
# 참조조문 텍스트에서 인용 패턴 추출 시도

pattern_analysis = []

for cat_key in ["precedent", "constitutional"]:
    cat_info = CATEGORIES[cat_key]
    cat_label = cat_info["label"]
    first_file = cat_info["files"][0]
    file_path = DATA_DIR / first_file
    
    if not file_path.exists():
        continue
    
    if USE_FULL_DATA:
        sample = load_all(file_path)
    else:
        sample = head_sample(file_path, n=SAMPLE_N)
    
    # 참조조문 필드
    ref_field = "참조조문"
    all_citations = []
    citations_per_record = []
    
    for rec in sample:
        text = rec.get(ref_field, "")
        if not isinstance(text, str) or not text.strip():
            citations_per_record.append(0)
            continue
        
        # 인용 추출
        cites = extract_citations(text)
        names = extract_law_names(text)
        combined = list(set(cites) | set(names))
        
        all_citations.extend(cites)
        citations_per_record.append(len(combined))
    
    # 통계
    arr = np.array(citations_per_record)
    has_citations = sum(1 for c in citations_per_record if c > 0)
    
    pattern_analysis.append({
        "카테고리": cat_label,
        "category": cat_key,
        "필드": ref_field,
        "인용 추출 성공": has_citations,
        "추출률 (%)": round(has_citations / len(sample) * 100, 1),
        "레코드당 평균": round(arr[arr > 0].mean(), 1) if has_citations > 0 else 0,
        "레코드당 최대": int(arr.max()),
        "고유 법령 수": len(set(all_citations)),
        "citations_per_record": citations_per_record,
    })
    
    print(f"\n{cat_label} - {ref_field}:")
    print(f"  추출 성공: {has_citations}/{len(sample)} ({has_citations/len(sample)*100:.1f}%)")
    print(f"  레코드당 평균: {arr[arr > 0].mean():.1f}" if has_citations > 0 else "  레코드당 평균: 0")
    print(f"  고유 법령: {len(set(all_citations))}개")
    
    # TOP 10 가장 많이 인용된 법령
    from collections import Counter
    top_cites = Counter(all_citations).most_common(10)
    print(f"  TOP 10 인용 법령:")
    for cite, cnt in top_cites:
        print(f"    {cite}: {cnt}회")

# 히스토그램: 레코드당 인용 수 분포
hist_data = []
for pa in pattern_analysis:
    for cnt in pa["citations_per_record"]:
        if cnt > 0:  # 인용이 있는 것만
            hist_data.append({"카테고리": pa["카테고리"], "인용 수": min(cnt, 30)})  # cap at 30

if hist_data:
    df_hist = pd.DataFrame(hist_data)
    fig = px.histogram(
        df_hist,
        x="인용 수",
        color="카테고리",
        title="레코드당 인용 법령 수 분포 (참조조문 필드)",
        nbins=30,
        barmode="overlay",
        opacity=0.7,
    )
    fig.update_layout(height=400)
    fig.show()


판례 - 참조조문:
  추출 성공: 66494/92055 (72.2%)
  레코드당 평균: 1.7
  고유 법령: 15106개
  TOP 10 인용 법령:
    민법 제750조: 2141회
    민법 제105조: 1470회
    행정소송법 제1조: 1187회
    민법 제2조: 1151회
    형법 제355조: 940회
    민법 제186조: 925회
    민사소송법 제187조: 871회
    민법 제763조: 851회
    민법 제393조: 787회
    국가배상법 제2조: 785회

헌재결정례 - 참조조문:
  추출 성공: 3738/31718 (11.8%)
  레코드당 평균: 1.7
  고유 법령: 2074개
  TOP 10 인용 법령:
    헌법 제11조: 909회
    헌법 제10조: 748회
    헌법재판소법 제68조: 299회
    헌법 제23조: 213회
    헌법 제12조: 190회
    항헌법재판소법 제68조: 185회
    헌법 제15조: 144회
    헌법 제27조: 109회
    헌법 제21조: 90회
    조헌법재판소법 제68조: 73회


In [5]:
# ── Cell 5: 비구조화 텍스트 내 법령 참조 (Medium Priority) ─
# 이유/결정요지 등 본문 텍스트에서 법령 참조 추출

UNSTRUCTURED_TARGETS = {
    "legislation": {"fields": ["이유"], "label": "법령해석례"},
    "committee": {"fields": ["이유", "결정요지"], "label": "위원회 결정문"},
    "administration": {"fields": ["이유"], "label": "행정심판례"},
    "special_tribunal": {"fields": ["이유"], "label": "특별행정심판"},
}

unstructured_results = []

for cat_key, target in UNSTRUCTURED_TARGETS.items():
    cat_info = CATEGORIES[cat_key]
    first_file = cat_info["files"][0]
    file_path = DATA_DIR / first_file
    
    if not file_path.exists():
        continue
    
    if USE_FULL_DATA:
        sample = load_all(file_path)
    else:
        sample = head_sample(file_path, n=SAMPLE_N)
    
    for field_name in target["fields"]:
        records_with_refs = 0
        total_refs = 0
        ref_counts = []
        
        for rec in sample:
            text = rec.get(field_name, "")
            if not isinstance(text, str) or not text.strip():
                ref_counts.append(0)
                continue
            
            citations = extract_citations(text)
            names = extract_law_names(text)
            combined = list(set([c.split(" 제")[0] for c in citations]) | set(names))
            
            if combined:
                records_with_refs += 1
                total_refs += len(combined)
            ref_counts.append(len(combined))
        
        extraction_rate = records_with_refs / len(sample) * 100 if sample else 0
        avg_refs = total_refs / records_with_refs if records_with_refs > 0 else 0
        
        unstructured_results.append({
            "카테고리": target["label"],
            "category": cat_key,
            "필드": field_name,
            "추출 성공 레코드": records_with_refs,
            "추출률 (%)": round(extraction_rate, 1),
            "평균 법령 수": round(avg_refs, 1),
            "유형": "비구조화 텍스트",
        })
        
        print(f"{target['label']} - {field_name}: {records_with_refs}/{len(sample)} ({extraction_rate:.1f}%), 평균 {avg_refs:.1f}개")

df_unstruct = pd.DataFrame(unstructured_results)
display(df_unstruct.style.format({
    "추출 성공 레코드": "{:,}",
    "추출률 (%)": "{:.1f}",
    "평균 법령 수": "{:.1f}",
}))

# 구조화 vs 비구조화 비교 바 차트
all_results = []
for sa in structured_analysis:
    all_results.append({
        "카테고리": sa["카테고리"],
        "필드": sa["필드"],
        "유효율 (%)": sa["유효율 (%)"],
        "유형": "구조화",
    })
for ur in unstructured_results:
    all_results.append({
        "카테고리": ur["카테고리"],
        "필드": ur["필드"],
        "유효율 (%)": ur["추출률 (%)"],
        "유형": "비구조화",
    })

df_all = pd.DataFrame(all_results)
fig = px.bar(
    df_all,
    x="카테고리",
    y="유효율 (%)",
    color="유형",
    barmode="group",
    title="인용 데이터 유효율: 구조화 vs 비구조화",
    hover_data=["필드"],
    color_discrete_map={"구조화": "#4472C4", "비구조화": "#ED7D31"},
    text="유효율 (%)",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(height=450, xaxis_tickangle=-30)
fig.show()

법령해석례 - 이유: 8591/8597 (99.9%), 평균 5.1개
위원회 결정문 - 이유: 635/635 (100.0%), 평균 5.5개
위원회 결정문 - 결정요지: 3/635 (0.5%), 평균 1.0개
행정심판례 - 이유: 33489/34254 (97.8%), 평균 4.0개
특별행정심판 - 이유: 12584/13846 (90.9%), 평균 1.9개


,카테고리,category,필드,추출 성공 레코드,추출률 (%),평균 법령 수,유형
0,법령해석례,legislation,이유,"8,591",99.9,5.1,비구조화 텍스트
1,위원회 결정문,committee,이유,635,100.0,5.5,비구조화 텍스트
2,위원회 결정문,committee,결정요지,3,0.5,1.0,비구조화 텍스트
3,행정심판례,administration,이유,"33,489",97.8,4.0,비구조화 텍스트
4,특별행정심판,special_tribunal,이유,"12,584",90.9,1.9,비구조화 텍스트


In [6]:
# ── Cell 6: 교차 참조 가능성 분석 ────────────────────────
# cgm_expc의 관련법령 필드에서 기존 Neo4j Statute 노드와 매칭 시도

# 먼저 cgm_expc 관련법령 샘플 로드
cgm_file = DATA_DIR / CATEGORIES["cgm_expc"]["files"][0]
if cgm_file.exists():
    if USE_FULL_DATA:
        cgm_sample = load_all(cgm_file)
    else:
        cgm_sample = head_sample(cgm_file, n=SAMPLE_N)
    
    # 관련법령 필드에서 법령명 추출
    cgm_law_names = []
    for rec in cgm_sample:
        val = rec.get("관련법령", "")
        if isinstance(val, str) and val.strip():
            # 관련법령은 보통 법령명 자체이거나 「법령명」 형태
            names = extract_law_names(val)
            if names:
                cgm_law_names.extend(names)
            else:
                # 「」가 없으면 전체를 법령명으로 간주 (쉼표/세미콜론 분리)
                for part in val.replace(";", ",").split(","):
                    part = part.strip()
                    if part and len(part) > 2:
                        cgm_law_names.append(part)
    
    from collections import Counter
    cgm_law_counter = Counter(cgm_law_names)
    
    print(f"cgm_expc 관련법령 분석 ({len(cgm_sample)}건 {'전체' if USE_FULL_DATA else '샘플'}):")
    print(f"  고유 법령명: {len(cgm_law_counter)}개")
    print(f"  총 참조: {sum(cgm_law_counter.values())}건")
    print(f"\n  TOP 15 가장 많이 참조된 법령:")
    for name, cnt in cgm_law_counter.most_common(15):
        print(f"    {name}: {cnt}회")
else:
    cgm_law_counter = Counter()
    print("cgm_expc 파일 없음")

# 교차 참조 분석: 판례와 부처 해석례가 공통으로 참조하는 법령
prec_file = DATA_DIR / CATEGORIES["precedent"]["files"][0]
if prec_file.exists():
    if USE_FULL_DATA:
        prec_sample = load_all(prec_file)
    else:
        prec_sample = head_sample(prec_file, n=SAMPLE_N)
    
    prec_law_names = []
    for rec in prec_sample:
        text = rec.get("참조조문", "")
        if isinstance(text, str) and text.strip():
            names = extract_law_names(text)
            prec_law_names.extend(names)
    
    prec_law_set = set(prec_law_names)
    cgm_law_set = set(cgm_law_counter.keys())
    
    common = prec_law_set & cgm_law_set
    
    print(f"\n교차 참조 분석:")
    print(f"  판례 참조 법령: {len(prec_law_set)}개")
    print(f"  부처 해석례 참조 법령: {len(cgm_law_set)}개")
    print(f"  공통 법령: {len(common)}개")
    if common:
        print(f"  공통 법령 예시: {list(common)[:10]}")

# 네트워크 효과 추정
print(f"\n네트워크 효과 추정:")
print(f"  현재 Neo4j: Statute 5,572 + Case 65,107 + 관계 163,854")
for cat_key in ["constitutional", "administration", "cgm_expc"]:
    total = cat_records.get(cat_key, 0)
    # 해당 카테고리가 추가되면 예상되는 관계 수
    est_edges = int(total * 2.5)  # 레코드당 평균 2.5개 관계 추정
    print(f"  + {CATEGORIES[cat_key]['label']} ({total:,}건): 예상 관계 +{est_edges:,}개")

cgm_expc 관련법령 분석 (528건 전체):
  고유 법령명: 271개
  총 참조: 532건

  TOP 15 가장 많이 참조된 법령:
    방위사업법: 40회
    표준화 업무규정: 24회
    무기체계 제안서 평가업무 지침: 13회
    국가를 당사자로 하는 계약에 관한 법률 시행령: 12회
    국방규격표준서의 서식 및 작성에 관한 매뉴얼: 11회
    국가를 당사자로 하는 계약에 관한 법률 시행령 제42조(국고의 부담이 되는 경쟁입찰에서의 낙찰자 결정): 10회
    국가를 당사자로 하는 계약에 관한 법률 시행령 제12조(경쟁입찰의 참가자격): 9회
    국가를 당사자로 하는 계약에 관한 법률 시행령 제39조(입찰서의 제출ㆍ접수 및 입찰의 무효): 9회
    국가를 당사자로 하는 계약에 관한 법률 시행령  제39조(입찰서의 제출ㆍ접수 및 입찰의 무효): 9회
    방위사업법 시행령  제33조(군수품목록정보): 8회
    방위사업법 시행령 제33조(군수품목록정보): 8회
    국가를 당사자로 하는 계약에 관한 법률 시행령  제12조(경쟁입찰의 참가자격): 8회
    국가를 당사자로 하는 계약에 관한 법률 시행령  제42조(국고의 부담이 되는 경쟁입찰에서의 낙찰자 결정): 8회
    전자정부법: 8회
    표준화업무규정: 7회

교차 참조 분석:
  판례 참조 법령: 10개
  부처 해석례 참조 법령: 271개
  공통 법령: 0개

네트워크 효과 추정:
  현재 Neo4j: Statute 5,572 + Case 65,107 + 관계 163,854
  + 헌재결정례 (31,718건): 예상 관계 +79,295개
  + 행정심판례 (34,254건): 예상 관계 +85,635개
  + 부처 해석례 (37,325건): 예상 관계 +93,312개


In [7]:
# ── Cell 7: Neo4j 확장 우선순위 매트릭스 ──────────────────

priority_scores = []

for cat_key, cat_info in CATEGORIES.items():
    cat_label = cat_info["label"]
    total = cat_records.get(cat_key, 0)
    score = 0
    reasons = []
    
    # 기준 1: 구조화된 인용 필드 존재 (+3점)
    has_structured = cat_key in STRUCTURED_TARGETS
    if has_structured:
        score += 3
        reasons.append("구조화 인용 필드 +3")
    
    # 기준 2: 인용 커버리지 50% 이상 (+2점)
    if has_structured:
        sa_entries = [s for s in structured_analysis if s["category"] == cat_key]
        max_rate = max((s["유효율 (%)"] for s in sa_entries), default=0)
        if max_rate >= 50:
            score += 2
            reasons.append(f"커버리지 {max_rate:.0f}% +2")
    else:
        # 비구조화 텍스트 추출률 확인
        ur_entries = [u for u in unstructured_results if u["category"] == cat_key]
        max_rate = max((u["추출률 (%)"] for u in ur_entries), default=0)
        if max_rate >= 50:
            score += 2
            reasons.append(f"텍스트 추출률 {max_rate:.0f}% +2")
    
    # 기준 3: 파싱 용이성 (+1점)
    # 구조화된 필드 또는 「」패턴이 있는 경우
    if has_structured or cat_key in ["legislation", "committee"]:
        score += 1
        reasons.append("파싱 용이 +1")
    
    # 기준 4: 레코드 수 10K 이상 (+1점)
    if total >= 10000:
        score += 1
        reasons.append(f"{total:,}건 +1")
    
    priority_scores.append({
        "카테고리": cat_label,
        "category": cat_key,
        "레코드 수": total,
        "점수": score,
        "근거": ", ".join(reasons) if reasons else "해당 없음",
    })

df_priority = pd.DataFrame(priority_scores).sort_values("점수", ascending=False)
display(df_priority.style.format({"레코드 수": "{:,}"}))

# 우선순위 시각화
fig = px.bar(
    df_priority.sort_values("점수"),
    x="점수",
    y="카테고리",
    orientation="h",
    title="Neo4j 확장 우선순위 (점수)",
    color="점수",
    color_continuous_scale="RdYlGn",
    text="점수",
    hover_data=["레코드 수", "근거"],
)
fig.update_traces(texttemplate="%{text}", textposition="outside")
fig.update_layout(height=500)
fig.show()

# 결론
print("\n" + "="*60)
print("Neo4j 확장 우선순위 결론")
print("="*60)
for i, (_, row) in enumerate(df_priority.iterrows()):
    if row["점수"] >= 5:
        tier = "Tier 1 (즉시 확장)"
    elif row["점수"] >= 3:
        tier = "Tier 2 (단기 확장)"
    elif row["점수"] >= 1:
        tier = "Tier 3 (중기 확장)"
    else:
        tier = "제외 (인용 데이터 부족)"
    print(f"  {row['카테고리']} (점수 {row['점수']}): {tier}")

,카테고리,category,레코드 수,점수,근거
0,판례,precedent,"92,055",7,"구조화 인용 필드 +3, 커버리지 80% +2, 파싱 용이 +1, 92,055건 +1"
7,부처 해석례,cgm_expc,"37,325",7,"구조화 인용 필드 +3, 커버리지 100% +2, 파싱 용이 +1, 37,325건 +1"
2,헌재결정례,constitutional,"31,718",5,"구조화 인용 필드 +3, 파싱 용이 +1, 31,718건 +1"
6,위원회 결정문,committee,"56,802",4,"텍스트 추출률 100% +2, 파싱 용이 +1, 56,802건 +1"
5,법령해석례,legislation,"8,597",3,"텍스트 추출률 100% +2, 파싱 용이 +1"
4,특별행정심판,special_tribunal,"148,778",3,"텍스트 추출률 91% +2, 148,778건 +1"
3,행정심판례,administration,"34,254",3,"텍스트 추출률 98% +2, 34,254건 +1"
8,법률용어사전,law_term,"81,488",1,"81,488건 +1"
1,법령,law,"5,548",0,해당 없음
9,조약,treaty,"3,589",0,해당 없음



Neo4j 확장 우선순위 결론
  판례 (점수 7): Tier 1 (즉시 확장)
  부처 해석례 (점수 7): Tier 1 (즉시 확장)
  헌재결정례 (점수 5): Tier 1 (즉시 확장)
  위원회 결정문 (점수 4): Tier 2 (단기 확장)
  법령해석례 (점수 3): Tier 2 (단기 확장)
  특별행정심판 (점수 3): Tier 2 (단기 확장)
  행정심판례 (점수 3): Tier 2 (단기 확장)
  법률용어사전 (점수 1): Tier 3 (중기 확장)
  법령 (점수 0): 제외 (인용 데이터 부족)
  조약 (점수 0): 제외 (인용 데이터 부족)
  행정규칙 (점수 0): 제외 (인용 데이터 부족)


In [8]:
# ── Cell 8: 결과 저장 ─────────────────────────────────────
result = {
    "analysis_type": "neo4j_citation_analysis",
    "sample_mode": "full" if USE_FULL_DATA else "dev",
    "sample_n": "all" if USE_FULL_DATA else SAMPLE_N,
    "inventory": [{k: v for k, v in d.items()} for d in inventory_data],
    "structured_analysis": structured_analysis,
    "pattern_analysis": [
        {k: v for k, v in pa.items() if k != "citations_per_record"}
        for pa in pattern_analysis
    ],
    "unstructured_analysis": unstructured_results,
    "priority_scores": [{k: v for k, v in ps.items()} for ps in priority_scores],
    "expansion_tiers": {
        "tier1_immediate": [
            ps["category"] for ps in priority_scores if ps["점수"] >= 5
        ],
        "tier2_short_term": [
            ps["category"] for ps in priority_scores if 3 <= ps["점수"] < 5
        ],
        "tier3_medium_term": [
            ps["category"] for ps in priority_scores if 1 <= ps["점수"] < 3
        ],
        "excluded": [
            ps["category"] for ps in priority_scores if ps["점수"] == 0
        ],
    },
}

path = save_result("phase6_citation_analysis", result)
print(f"저장 완료: {path}")

저장 완료: C:\Users\fkjy1\dev\boot_camp\law-3\backend\eda_output\phase6_citation_analysis.json
